# NCERT Class 6–10 inventory and EDA
Run privately on Kaggle with Internet enabled. This notebook retains each downloaded PDF, records its hash, checks page counts and reports gaps. Keep the output private.

In [1]:
%pip -q install pymupdf requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json, time, zipfile
from pathlib import Path
from urllib.parse import urlparse
import fitz, requests

ROOT = Path('/kaggle/working/ncert_inventory')
PDFS = ROOT / 'pdfs'
PDFS.mkdir(parents=True, exist_ok=True)
# Candidate book codes are checked against official PDF responses. Missing books remain gaps.
BOOKS = {
    (6, 'maths'): ['fegp1', 'femh1'], (6, 'science'): ['fecu1', 'fesc1'],
    (7, 'maths'): ['gegp1', 'gemh1'], (7, 'science'): ['gecu1', 'gesc1'],
    (8, 'maths'): ['hegp1', 'hemh1'], (8, 'science'): ['hecu1', 'hesc1'],
    (9, 'maths'): ['iemh1', 'kemh1'], (9, 'science'): ['iesc1', 'kesc1'],
    (10, 'maths'): ['jemh1'], (10, 'science'): ['jesc1'],
}
PART_TWOS = {(7, 'maths'): ('gegp2', 'gegp1', 8, 15),
             (8, 'maths'): ('hegp2', 'hegp1', 7, 14),
             (9, 'maths'): ('iemh2', 'iemh1', 8, 14)}
BASE = 'https://ncert.nic.in/textbook/pdf/'
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 NCERT-research-inventory'})

def fetch(code):
    url = BASE + code + '.pdf'
    for attempt in range(3):
        try:
            # shortcut: NCERT TLS chain fails on Kaggle; verify host and PDF bytes, upgrade when TLS is fixed.
            response = session.get(url, timeout=60, verify=False)
            if response.status_code == 404: return None
            response.raise_for_status()
            if urlparse(response.url).hostname not in {'ncert.nic.in', 'www.ncert.nic.in'}: raise ValueError('Unexpected redirect')
            if not response.content.startswith(b'%PDF-'): raise ValueError('Response is not a PDF')
            return response.content
        except requests.RequestException:
            if attempt == 2: raise
            time.sleep(2 ** attempt)


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
records, gaps = [], []
for (grade, subject), candidates in BOOKS.items():
    selected = None
    for prefix in candidates:
        prelims = fetch(prefix + 'ps')
        if prelims:
            selected = prefix
            break
    if selected is None:
        gaps.append({'grade': grade, 'subject': subject, 'reason': 'No candidate prelims found', 'candidates': candidates})
        continue
    found = 0
    for number in range(1, 21):
        code = f'{selected}{number:02d}'
        data = fetch(code)
        if data is None: continue
        document = fitz.open(stream=data, filetype='pdf')
        path = PDFS / f'{grade}_{subject}_{code}.pdf'
        path.write_bytes(data)
        records.append({'grade': grade, 'subject': subject, 'book_code': selected, 'chapter': number, 'url': BASE + code + '.pdf', 'file': path.name, 'sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data), 'pages': len(document), 'first_page_text_chars': len(document[0].get_text())})
        found += 1
    if not found: gaps.append({'grade': grade, 'subject': subject, 'reason': 'Book found but no chapters found', 'book_code': selected})
    if (grade, subject) in PART_TWOS and selected == PART_TWOS[grade, subject][1]:
        part_two, _, offset, expected = PART_TWOS[grade, subject]
        if fetch(part_two + 'ps') is None:
            gaps.append({'grade': grade, 'subject': subject, 'reason': 'Part II prelims unavailable', 'book_code': part_two})
        else:
            for number in range(1, 21):
                code = f'{part_two}{number:02d}'
                data = fetch(code)
                if data is None: continue
                document = fitz.open(stream=data, filetype='pdf')
                path = PDFS / f'{grade}_{subject}_{code}.pdf'
                path.write_bytes(data)
                records.append({'grade': grade, 'subject': subject, 'book_code': part_two, 'chapter': offset + number, 'url': BASE + code + '.pdf', 'file': path.name, 'sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data), 'pages': len(document), 'first_page_text_chars': len(document[0].get_text())})
                found += 1
        if found < expected: gaps.append({'grade': grade, 'subject': subject, 'reason': 'Expected 2026 edition chapters missing', 'missing': sorted(set(range(1, expected + 1)) - {r['chapter'] for r in records if r['grade'] == grade and r['subject'] == subject})})
    elif (grade, subject) in PART_TWOS:
        gaps.append({'grade': grade, 'subject': subject, 'reason': 'Current two-part edition unavailable; older book selected', 'book_code': selected})
    print(grade, subject, selected, found, 'chapters')
    (ROOT / 'manifest.json').write_text(json.dumps(records, indent=2))
    (ROOT / 'gaps.json').write_text(json.dumps(gaps, indent=2))
assert len({r['file'] for r in records}) == len(records)
assert all(r['pages'] > 0 and r['bytes'] > 1000 for r in records)
print('Total chapters:', len(records), 'Total pages:', sum(r['pages'] for r in records), 'Gaps:', len(gaps))


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: Inse

6 maths fegp1 10 chapters


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: Inse

6 science fecu1 12 chapters


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: Inse

7 maths gegp1 15 chapters


... (10 outputs left out)

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: Inse

10 maths jemh1 14 chapters


/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ncert.nic.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1129: Inse

10 science jesc1 13 chapters
Total chapters: 130 Total pages: 3066 Gaps: 0


In [4]:
# Save a single versioned artifact for the next notebook. Keep this notebook private.
archive = ROOT / 'ncert_source_pdfs.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
    for row in records:
        output.write(PDFS / row['file'], row['file'])
print(archive, archive.stat().st_size, 'bytes')


/kaggle/working/ncert_inventory/ncert_source_pdfs.zip 520351835 bytes
